# Gold v2 Reference — Luxia only resume

Luxia Reference 3조건만 이어서 실행하는 notebook입니다.

- 기존 HF4 결과는 절대 다시 돌리지 않습니다.
- Drive `_luxia_finalize_tmp`의 Luxia prediction JSONL을 그대로 재사용합니다.
- 성공한 행은 건너뛰고, API error 행 + 아직 호출하지 않은 행만 재시도합니다.
- `CHECKPOINT_EVERY=1`이라 한 문제마다 Drive에 저장됩니다.
- 완료 후 기존 HF 12조건과 merge하여 최종 5 LVLM × 3 Reference = 15조건을 만듭니다.


In [ ]:
REPO='arnold6444/lvlm_insplad'
REPO_REF='main'
QA_FOLDER='lvlm_insplad_qa/gold_v2_final'
BASELINE_FOLDER='lvlm_insplad_results/gold_baseline_v2'
REFERENCE_FOLDER='lvlm_insplad_results/gold_reference_v2'
TMP_FOLDER='lvlm_insplad_results/gold_reference_v2/_luxia_finalize_tmp'
CONDITIONS=['normal_reference','positive_reference','normal_positive_reference']
HF_MODELS=['qwen3_5_4b','internvl3_5_8b','minicpm_v_4_5','llava_onevision_7b']
ALL_MODELS=HF_MODELS+['luxia']
CHECKPOINT_EVERY=1
UNSUP_ZIP_NAME='unsupervised_anomaly_detection.zip'
UNSUP_ZIP_SIZE=1_170_967_719
SUP_ZIP_NAME='supervised_fault_classification.zip'
SUP_ZIP_SIZE=874_753_167


In [ ]:
import base64,csv,json,os,shutil,subprocess,sys
from pathlib import Path
from google.colab import drive,userdata
drive.mount('/content/drive', force_remount=False)
DRIVE_ROOT=Path('/content/drive/MyDrive')
QA_DIR=DRIVE_ROOT/QA_FOLDER
BASELINE_DIR=DRIVE_ROOT/BASELINE_FOLDER
RESULT_DIR=DRIVE_ROOT/REFERENCE_FOLDER
TMP_DIR=DRIVE_ROOT/TMP_FOLDER
RESULT_DIR.mkdir(parents=True,exist_ok=True); TMP_DIR.mkdir(parents=True,exist_ok=True)
REPO_DIR=Path('/content/lvlm_insplad')
if REPO_DIR.exists(): shutil.rmtree(REPO_DIR)
def run(cmd,*,env=None,check=True):
    cmd=[str(x) for x in cmd]; print('+',' '.join(cmd[:12])+(' ...' if len(cmd)>12 else ''))
    return subprocess.run(cmd,env=env,check=check)
github_token=userdata.get('GITHUB_TOKEN')
if not github_token: raise RuntimeError('Colab Secret GITHUB_TOKEN이 필요합니다.')
os.environ['GITHUB_TOKEN']=github_token
luxia_key=userdata.get('LUXIA_API_KEY')
if not luxia_key: raise RuntimeError('Colab Secret LUXIA_API_KEY가 필요합니다.')
os.environ['LUXIA_API_KEY']=luxia_key
auth=base64.b64encode(f'x-access-token:{github_token}'.encode()).decode()
git_env=os.environ.copy(); git_env.update({'GIT_CONFIG_COUNT':'1','GIT_CONFIG_KEY_0':'http.extraHeader','GIT_CONFIG_VALUE_0':f'AUTHORIZATION: basic {auth}','GIT_TERMINAL_PROMPT':'0'})
run(['git','clone','--depth','1','--branch',REPO_REF,f'https://github.com/{REPO}.git',REPO_DIR],env=git_env)
run([sys.executable,'-m','pip','install','-q','pandas>=2.2','numpy>=1.26','pillow>=10','tqdm>=4.66'])
run([sys.executable,'-m','pip','install','-q','-e',f'{REPO_DIR}[dev]'])


In [ ]:
# 입력 파일 확인
def find_unique_sized(filename,expected_size):
    xs=[p for p in DRIVE_ROOT.rglob(filename) if p.is_file() and p.stat().st_size==expected_size]
    if not xs: raise FileNotFoundError(f'{filename} ({expected_size} bytes)')
    return xs[0]
SUP_ZIP=find_unique_sized(SUP_ZIP_NAME,SUP_ZIP_SIZE)
UNSUP_ZIP=find_unique_sized(UNSUP_ZIP_NAME,UNSUP_ZIP_SIZE)
BASELINE_LEADERBOARD=BASELINE_DIR/'leaderboard.csv'
positive_candidates=[RESULT_DIR/'positive_reference_build'/'positive_reference_manifest.csv', RESULT_DIR/'positive_reference_manifest.csv']
POSITIVE_MANIFEST=next((p for p in positive_candidates if p.is_file()),None)
if POSITIVE_MANIFEST is None:
    found=list(RESULT_DIR.rglob('positive_reference_manifest.csv'))
    if not found: raise FileNotFoundError('positive_reference_manifest.csv')
    POSITIVE_MANIFEST=found[0]
import pandas as pd
existing=pd.read_csv(RESULT_DIR/'leaderboard.csv')
hf=existing[existing.model_key.isin(HF_MODELS)&existing.condition.isin(CONDITIONS)].drop_duplicates(['model_key','condition'])
if len(hf)!=12: raise RuntimeError(f'HF reference coverage {len(hf)}/12')
print('HF4 기존 결과 12/12 확인 — 재실행 안 함')


In [ ]:
# 현재 Luxia 진행률
def read_jsonl(path):
    if not path.is_file(): return []
    return [json.loads(x) for x in path.read_text(encoding='utf-8').splitlines() if x.strip()]
qa=read_jsonl(TMP_DIR/'run_qa.jsonl')
if not qa: qa=read_jsonl(RESULT_DIR/'run_qa.jsonl')
expected=len(qa)
print('questions per condition:',expected)
total_pending=0
for c in CONDITIONS:
    p=TMP_DIR/c/'predictions'/'luxia.jsonl'
    rows=read_jsonl(p); by={str(r.get('id')):r for r in rows}
    reusable={k for k,r in by.items() if not r.get('error') and str(r.get('output','')).strip()}
    pending=expected-len(reusable)
    total_pending+=pending
    errors=sum(bool(r.get('error')) for r in rows)
    print(f'{c}: reusable={len(reusable)}/{expected}, errors_to_retry={errors}, pending_calls={pending}')
print('TOTAL pending Luxia calls:',total_pending)


In [ ]:
# Luxia 3조건만 resume — 성공 prediction은 자동 재사용
luxia_env=os.environ.copy(); luxia_env.pop('GITHUB_TOKEN',None)
run([sys.executable,REPO_DIR/'scripts/run_gold_v2_reference_ablation.py',
     '--qa-dir',QA_DIR,'--sup-zip',SUP_ZIP,'--unsup-zip',UNSUP_ZIP,
     '--positive-manifest',POSITIVE_MANIFEST,'--out-dir',TMP_DIR,
     '--baseline-leaderboard',BASELINE_LEADERBOARD,
     '--checkpoint-every',str(CHECKPOINT_EVERY),'--conditions',*CONDITIONS,'--models','luxia'],env=luxia_env)


In [ ]:
# 완료 검증: error/미호출 0개가 될 때까지 merge하지 않음
qa=read_jsonl(TMP_DIR/'run_qa.jsonl'); expected=len(qa)
pending_total=0
for c in CONDITIONS:
    rows=read_jsonl(TMP_DIR/c/'predictions'/'luxia.jsonl')
    reusable=sum(1 for r in rows if not r.get('error') and str(r.get('output','')).strip())
    pending=expected-reusable; pending_total+=pending
    print(c,reusable,'/',expected,'pending',pending)
if pending_total:
    raise RuntimeError(f'Luxia 아직 {pending_total}문항 남음. notebook 재실행하면 이어서 재시도합니다.')
print('Luxia 3조건 전부 완료')


In [ ]:
# 기존 HF12 + Luxia3 merge → 최종 15조건
def read_csv_rows(path):
    if not path.is_file() or not path.stat().st_size: return []
    with path.open(newline='',encoding='utf-8-sig') as f: return list(csv.DictReader(f))
def write_csv_rows(path,rows):
    fields=[]
    for r in rows:
        for k in r:
            if k not in fields: fields.append(k)
    with path.open('w',newline='',encoding='utf-8') as f:
        w=csv.DictWriter(f,fieldnames=fields); w.writeheader(); w.writerows(rows)
def merge_rows(old,new,keys):
    repl={tuple(str(r.get(k,'')) for k in keys) for r in new}
    return [r for r in old if tuple(str(r.get(k,'')) for k in keys) not in repl]+new
write_csv_rows(RESULT_DIR/'leaderboard.csv',merge_rows(read_csv_rows(RESULT_DIR/'leaderboard.csv'),read_csv_rows(TMP_DIR/'leaderboard.csv'),('model_key','condition')))
write_csv_rows(RESULT_DIR/'metrics_by_asset.csv',merge_rows(read_csv_rows(RESULT_DIR/'metrics_by_asset.csv'),read_csv_rows(TMP_DIR/'metrics_by_asset.csv'),('model_key','condition','asset_category')))
for c in CONDITIONS:
    src=TMP_DIR/c/'predictions'/'luxia.jsonl'; dst=RESULT_DIR/c/'predictions'/'luxia.jsonl'
    dst.parent.mkdir(parents=True,exist_ok=True); shutil.copy2(src,dst)
import importlib.util
spec=importlib.util.spec_from_file_location('ref_runner',REPO_DIR/'scripts/run_gold_v2_reference_ablation.py')
mod=importlib.util.module_from_spec(spec); spec.loader.exec_module(mod)
comparison,deltas=mod.build_ablation_comparison(read_csv_rows(RESULT_DIR/'leaderboard.csv'),BASELINE_LEADERBOARD)
write_csv_rows(RESULT_DIR/'ablation_comparison.csv',comparison); write_csv_rows(RESULT_DIR/'ablation_vs_baseline.csv',deltas)
final=pd.read_csv(RESULT_DIR/'leaderboard.csv')
coverage=final[final.model_key.isin(ALL_MODELS)&final.condition.isin(CONDITIONS)].drop_duplicates(['model_key','condition'])
if len(coverage)!=15: raise RuntimeError(f'coverage {len(coverage)}/15')
meta={'benchmark_version':'gold_v2_final','gold_images':246,'conditions':CONDITIONS,'models':ALL_MODELS,'expected_runs':15,'status':'complete','note':'HF4 existing results reused; Luxia-only resume finalized separately.'}
(RESULT_DIR/'run_metadata.json').write_text(json.dumps(meta,ensure_ascii=False,indent=2)+'\n',encoding='utf-8')
display(coverage.sort_values(['condition','model_key']))


In [ ]:
# GitHub publish
files=['leaderboard.csv','metrics_by_asset.csv','ablation_comparison.csv','ablation_vs_baseline.csv','reference_manifest.csv','positive_reference_summary.csv','positive_reference_metadata.json','run_status.json','run_metadata.json']
run([sys.executable,REPO_DIR/'scripts/publish_results_to_github.py','--out-dir',RESULT_DIR,'--repo',REPO,'--branch',REPO_REF,'--prefix','gold_v2_reference','--files',*[f for f in files if (RESULT_DIR/f).is_file()]],env=os.environ.copy())
print('DONE: 5 LVLM × 3 Reference = 15조건')
